# Seed Synthesizability

Runs AiZynthFinder on the two upstream molecule layers to measure how synthesizability
changes across the pipeline:

```
ChEMBL (spec_smiles)  →  PrexSyn  →  seed_smiles  →  PrexSyn resample  →  baseline variants
       [this notebook]              [this notebook]
```

**Output:** `data/generation_stratified/synth_seeds_depth{N}.json`  
Maps each SMILES → bool (synthesizable at depth N).  
Results are also written to `data/generation_stratified/seed_synth_results.csv`
for use in `results_tables.ipynb`.

In [ ]:
import sys, json
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, TimeoutError as FutureTimeoutError, as_completed
from concurrent.futures.process import BrokenProcessPool

import pandas as pd
from tqdm.notebook import tqdm

ROOT = Path('.').resolve()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.evaluation.synth_parallel import worker_init, score_one

# ── Config ────────────────────────────────────────────────────────────────────
GEN_DIR    = ROOT / 'data' / 'generation_stratified'
SEEDS_FILE = GEN_DIR / 'seeds_for_methods_stratified.json'

AIZYNTHFINDER_CONFIG = ROOT / 'data' / 'aizynthfinder' / 'config.yml'

MAX_DEPTH  = 6
TIME_LIMIT = 120   # seconds per molecule
N_WORKERS  = 10
TIMEOUT    = TIME_LIMIT + 30

SYNTH_CKPT = GEN_DIR / f'synth_seeds_depth{MAX_DEPTH}.json'
RESULTS_CSV = GEN_DIR / 'seed_synth_results.csv'

print(f'Seeds file : {"OK" if SEEDS_FILE.exists() else "MISSING"}')
print(f'AiZynth cfg: {"OK" if AIZYNTHFINDER_CONFIG.exists() else "MISSING"}')
print(f'Max depth  : {MAX_DEPTH}')
print(f'Checkpoint : {SYNTH_CKPT.name} -> {"exists (will resume)" if SYNTH_CKPT.exists() else "fresh run"}')


---
## Stage 1 — Load seeds, collect unique molecules to score

In [ ]:
seeds_raw = json.load(open(SEEDS_FILE))
print(f'Seeds loaded: {len(seeds_raw)}')
print(f'Keys: {list(seeds_raw[0].keys())}')
print()

# Build per-seed rows with both SMILES and metadata
seed_records = []
for s in seeds_raw:
    seed_records.append({
        'spec_smiles':      s['spec_smiles'],
        'seed_smiles':      s['seed_smiles'],
        'quality_bin':      s['quality_bin'],
        'baseline_quality': s['baseline_quality'],
    })
seeds_df = pd.DataFrame(seed_records)

# Unique molecules to score (spec + seed, deduplicated)
spec_smiles = seeds_df['spec_smiles'].unique().tolist()
seed_smiles = seeds_df['seed_smiles'].unique().tolist()
all_smiles  = list(set(spec_smiles + seed_smiles))

n_identity = sum(1 for s in seeds_raw if s['spec_smiles'] == s['seed_smiles'])

print(f'Unique spec_smiles : {len(spec_smiles)}')
print(f'Unique seed_smiles : {len(seed_smiles)}')
print(f'Perfect reconstruction (spec==seed): {n_identity}  '
      f'<- PrexSyn decoded the same SMILES as the input for these {n_identity} seeds '
      f'(molecule sits at the center of its latent neighbourhood)')
print(f'Total unique to score: {len(all_smiles)}  (= 100 + 100 - {n_identity} deduplicated)')


---
## Stage 2 — Run AiZynthFinder (with checkpoint resuming)

In [ ]:
assert AIZYNTHFINDER_CONFIG.exists(), f'AiZynthFinder config not found: {AIZYNTHFINDER_CONFIG}'

synth: dict[str, bool] = {}
if SYNTH_CKPT.exists():
    with open(SYNTH_CKPT) as f:
        synth = json.load(f)
    print(f'Resumed checkpoint: {len(synth)} already scored')

pending = [s for s in all_smiles if s not in synth]
print(f'Remaining: {len(pending)} molecules to score')

if pending:
    with ProcessPoolExecutor(
        max_workers=N_WORKERS,
        initializer=worker_init,
        initargs=(str(AIZYNTHFINDER_CONFIG), MAX_DEPTH, TIME_LIMIT),
        max_tasks_per_child=None,
    ) as pool:
        futures = {pool.submit(score_one, s): s for s in pending}
        with tqdm(total=len(pending), desc=f'depth={MAX_DEPTH}', unit='mol') as pbar:
            for fut in as_completed(futures):
                smi = futures[fut]
                try:
                    _, solved = fut.result(timeout=TIMEOUT)
                except FutureTimeoutError:
                    solved = False
                except BrokenProcessPool:
                    with open(SYNTH_CKPT, 'w') as f:
                        json.dump(synth, f)
                    raise
                except Exception:
                    solved = False
                synth[smi] = solved
                pbar.update(1)

    with open(SYNTH_CKPT, 'w') as f:
        json.dump(synth, f)
    print(f'Checkpoint saved -> {SYNTH_CKPT}')

n_solved = sum(synth.values())
print(f'\nDone: {n_solved}/{len(synth)} synthesizable ({100*n_solved/max(len(synth),1):.1f}%)')


---
## Stage 3 — Attach results & save CSV

In [ ]:
col = f'is_synth_depth{MAX_DEPTH}'

seeds_df[f'spec_{col}'] = seeds_df['spec_smiles'].map(synth)
seeds_df[f'seed_{col}'] = seeds_df['seed_smiles'].map(synth)

# Molecules not in synth dict were never reached (shouldn't happen after a complete run)
missing_spec = seeds_df[f'spec_{col}'].isna().sum()
missing_seed = seeds_df[f'seed_{col}'].isna().sum()
if missing_spec or missing_seed:
    print(f'WARNING: {missing_spec} spec_smiles and {missing_seed} seed_smiles not in checkpoint')

seeds_df[f'spec_{col}'] = seeds_df[f'spec_{col}'].fillna(False)
seeds_df[f'seed_{col}'] = seeds_df[f'seed_{col}'].fillna(False)

seeds_df.to_csv(RESULTS_CSV, index=False)
print(f'Saved -> {RESULTS_CSV}')
print(seeds_df[[f'spec_{col}', f'seed_{col}']].value_counts().sort_index())


---
## Stage 4 — Analysis: synthesizability across pipeline layers

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

BIN_ORDER  = ['<0.5', '0.5-0.7', '0.7-0.85', '0.85-1.0']
BIN_COLORS = {'<0.5': '#e8c5a0', '0.5-0.7': '#a8d5a2', '0.7-0.85': '#6ab0de', '0.85-1.0': '#9b7dd4'}
col        = f'is_synth_depth{MAX_DEPTH}'

seeds_df['quality_bin'] = pd.Categorical(seeds_df['quality_bin'], categories=BIN_ORDER, ordered=True)

# ── Summary table ───────────────────────────────────────────────────────────────
rows = []
for b in BIN_ORDER:
    sub = seeds_df[seeds_df['quality_bin'] == b]
    n   = len(sub)
    spec_synth = sub[f'spec_{col}'].sum()
    seed_synth = sub[f'seed_{col}'].sum()
    rows.append({
        'Quality Bin':        b,
        'N':                  n,
        'ChEMBL synth':       int(spec_synth),
        'ChEMBL synth (%)':   round(spec_synth / n * 100, 1),
        'PrexSyn synth':      int(seed_synth),
        'PrexSyn synth (%)':  round(seed_synth / n * 100, 1),
        'Δ (PrexSyn - ChEMBL)': round((seed_synth - spec_synth) / n * 100, 1),
    })

summary = pd.DataFrame(rows)

# Overall row
overall = {
    'Quality Bin':        'Overall',
    'N':                  len(seeds_df),
    'ChEMBL synth':       int(seeds_df[f'spec_{col}'].sum()),
    'ChEMBL synth (%)':   round(seeds_df[f'spec_{col}'].mean() * 100, 1),
    'PrexSyn synth':      int(seeds_df[f'seed_{col}'].sum()),
    'PrexSyn synth (%)':  round(seeds_df[f'seed_{col}'].mean() * 100, 1),
    'Δ (PrexSyn - ChEMBL)': round((seeds_df[f'seed_{col}'].mean() - seeds_df[f'spec_{col}'].mean()) * 100, 1),
}
summary = pd.concat([summary, pd.DataFrame([overall])], ignore_index=True)

print(f'Synthesizability across pipeline layers  (AiZynthFinder depth {MAX_DEPTH})\n')
print(summary.to_string(index=False))


In [ ]:
# ── Figure ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
plot_bins = BIN_ORDER  # exclude Overall row
x         = np.arange(len(plot_bins))
colors    = [BIN_COLORS[b] for b in plot_bins]
w         = 0.35

spec_rates = [summary.loc[summary['Quality Bin']==b, 'ChEMBL synth (%)'].values[0]  for b in plot_bins]
seed_rates = [summary.loc[summary['Quality Bin']==b, 'PrexSyn synth (%)'].values[0] for b in plot_bins]
deltas     = [summary.loc[summary['Quality Bin']==b, 'Δ (PrexSyn - ChEMBL)'].values[0] for b in plot_bins]

# Left: grouped bar — ChEMBL vs PrexSyn synthesizability rate
b1 = axes[0].bar(x - w/2, spec_rates, w, label='ChEMBL (spec)',  color='#90CAF9', edgecolor='white')
b2 = axes[0].bar(x + w/2, seed_rates, w, label='PrexSyn (seed)', color='#A5D6A7', edgecolor='white')
for bar, v in zip(list(b1) + list(b2), spec_rates + seed_rates):
    axes[0].text(bar.get_x() + bar.get_width()/2, v + 0.5,
                 f'{v:.0f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(plot_bins, rotation=15)
axes[0].set_ylabel('Synthesizable (%)')
axes[0].set_title(f'Synthesizability: ChEMBL vs PrexSyn\n(AiZynthFinder depth {MAX_DEPTH})',
                  fontweight='bold')
axes[0].set_ylim(0, 110)
axes[0].legend(fontsize=9, frameon=False)
axes[0].grid(True, axis='y', alpha=0.25)

# Right: delta (PrexSyn - ChEMBL) per quality bin
bar_colors = ['#EF9A9A' if d < 0 else '#A5D6A7' for d in deltas]
bars = axes[1].bar(x, deltas, color=bar_colors, width=0.5, edgecolor='white')
for bar, v in zip(bars, deltas):
    va = 'bottom' if v >= 0 else 'top'
    offset = 0.3 if v >= 0 else -0.3
    axes[1].text(bar.get_x() + bar.get_width()/2, v + offset,
                 f'{v:+.1f}%', ha='center', va=va, fontsize=9, fontweight='bold')
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_xticks(x)
axes[1].set_xticklabels(plot_bins, rotation=15)
axes[1].set_ylabel('Δ synthesizability (PrexSyn − ChEMBL) (%)')
axes[1].set_title('PrexSyn Effect on Synthesizability\n(green = improved, red = degraded)',
                  fontweight='bold')
axes[1].grid(True, axis='y', alpha=0.25)

fig.suptitle(
    f'Pipeline Synthesizability: ChEMBL → PrexSyn  (depth {MAX_DEPTH}, n=100 seeds)',
    fontweight='bold', y=1.02,
)
plt.tight_layout()
out_path = GEN_DIR / f'fig_seed_synth_depth{MAX_DEPTH}.png'
plt.savefig(out_path, bbox_inches='tight', dpi=150)
plt.show()
print(f'Saved -> {out_path}')


---
## Stage 5 — Per-molecule breakdown (transitions)

For each seed, classify the ChEMBL → PrexSyn transition:

| ChEMBL synth | PrexSyn synth | Label |
|---|---|---|
| ✓ | ✓ | preserved |
| ✓ | ✗ | degraded |
| ✗ | ✓ | recovered |
| ✗ | ✗ | both non-synth |

In [ ]:
def transition(row):
    s, p = row[f'spec_{col}'], row[f'seed_{col}']
    if s and p:   return 'preserved'
    if s and not p: return 'degraded'
    if not s and p: return 'recovered'
    return 'both non-synth'

seeds_df['transition'] = seeds_df.apply(transition, axis=1)

TRANSITION_ORDER  = ['preserved', 'degraded', 'recovered', 'both non-synth']
TRANSITION_COLORS = {'preserved': '#A5D6A7', 'degraded': '#EF9A9A',
                     'recovered': '#90CAF9', 'both non-synth': '#E0E0E0'}

trans_summary = (
    seeds_df
    .groupby(['quality_bin', 'transition'], observed=True)
    .size()
    .unstack('transition', fill_value=0)
    .reindex(columns=TRANSITION_ORDER, fill_value=0)
)
trans_summary['Total'] = trans_summary.sum(axis=1)
print('Transition counts by quality bin:')
print(trans_summary.to_string())

# Stacked bar
fig, ax = plt.subplots(figsize=(9, 5))
bottom = np.zeros(len(BIN_ORDER))
for t in TRANSITION_ORDER:
    vals = [trans_summary.loc[b, t] if b in trans_summary.index else 0 for b in BIN_ORDER]
    bars = ax.bar(BIN_ORDER, vals, bottom=bottom,
                  label=t, color=TRANSITION_COLORS[t], edgecolor='white', width=0.5)
    for i, (bar, v) in enumerate(zip(bars, vals)):
        if v > 0:
            ax.text(bar.get_x() + bar.get_width()/2,
                    bottom[i] + v / 2,
                    str(v), ha='center', va='center', fontsize=9, fontweight='bold')
    bottom += np.array(vals, dtype=float)

ax.set_ylabel('Number of seeds')
ax.set_xlabel('Baseline quality bin')
ax.set_title(f'ChEMBL → PrexSyn Synthesizability Transitions\n(depth {MAX_DEPTH}, n=100)',
             fontweight='bold')
ax.legend(fontsize=9, frameon=False, bbox_to_anchor=(1.01, 1), loc='upper left')
ax.set_ylim(0, 30)
plt.tight_layout()
out_path2 = GEN_DIR / f'fig_seed_transitions_depth{MAX_DEPTH}.png'
plt.savefig(out_path2, bbox_inches='tight', dpi=150)
plt.show()
print(f'Saved -> {out_path2}')
